# 02 — Reusable modeling features
Requires notebook 01 checkpoint. No globals are inherited from another notebook.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.json").exists() and (p / "src" / "common.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.common import paths, save_frame, load_frame
config, project_paths = paths(ROOT)
data_dir = project_paths["raw"]
processed_dir = project_paths["processed"]
figures_dir = project_paths["figures"]
tables_dir = project_paths["tables"]
artifacts_dir = project_paths["artifacts"]
cutoff = pd.Timestamp(config["cutoff"])


In [ ]:
from src.features import add_features, model_inputs, NUMERIC_FEATURES, CATEGORICAL_FEATURES
base = load_frame(processed_dir / 'subscriber_base.csv.gz')
campaign = add_features(base.loc[base.campaign_eligible].copy(), cutoff)
X, y = model_inputs(campaign)
print('Feature table:', X.shape)
display(X[NUMERIC_FEATURES].isna().sum())
save_frame(campaign, processed_dir / 'modeling_table.csv.gz')
(tables_dir / 'feature_schema.json').write_text(json.dumps({'numeric': NUMERIC_FEATURES, 'categorical': CATEGORICAL_FEATURES}, indent=2))


The modeling checkpoint retains `msno`, row identity, and the payment proxy for later ID-aligned evaluation. `model_inputs()` selects only 21 allowed features. Revenue multiplied by observed churn is never a model input. See `docs/feature_dictionary.md` and `docs/review.md`.